# Muice N-Shot Stability Experiment

This notebook evaluates the cosine-similarity convergence behavior between the style vector extracted from sampled Muice data $S_N$ and the full-corpus reference vector $S_{full}$.

## 1. Install and Import Dependencies (Muice, BGE, Vector Ops, Plotting)

Install and import all required dependencies, then set random seeds and device configuration for reproducibility.

In [ ]:
# Uncomment to install if needed
# %pip install -q pandas numpy torch sentence-transformers scikit-learn matplotlib seaborn tqdm

import json
import random
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
sns.set_theme(style="whitegrid")


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for p in [current, *current.parents]:
        if (p / "OtakuLab").exists() and (p / "Models").exists():
            return p
    raise FileNotFoundError("Project root containing OtakuLab/ and Models/ was not found")


PROJECT_ROOT = find_project_root(Path.cwd())
OTAKU_ROOT = PROJECT_ROOT / "OtakuLab"
OUTPUT_ROOT = OTAKU_ROOT / "outputs"
RESULT_ROOT = OUTPUT_ROOT / "stability_nshot"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

TEXT_PATH = OUTPUT_ROOT / "rag" / "Muice_corpus.jsonl"
PMI_PATH = OUTPUT_ROOT / "pmi" / "muice_pmi_filtered.json"
PRAG_PATH = OUTPUT_ROOT / "pragmatic" / "muice.jsonl"
CONS_PATH = OUTPUT_ROOT / "cons" / "muice.json"

BGE_LOCAL = PROJECT_ROOT / "Models" / "bge-large-zh-v1.5"
BGE_MODEL_NAME = str(BGE_LOCAL) if BGE_LOCAL.exists() else "BAAI/bge-large-zh-v1.5"
print(f"Project root: {PROJECT_ROOT}")
print(f"BGE model: {BGE_MODEL_NAME}")

## 2. Load Muice Data and Build the Full-Corpus Reference Vector

Load Muice resources and standardize fields (text, keyword scores, pragmatic labels, and constituency parse trees).

In [11]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


# Muice texts
muice_rows = read_jsonl(TEXT_PATH)
muice_texts = [r.get("text", "").strip() for r in muice_rows if r.get("text", "").strip()]
muice_text_set = set(muice_texts)

# PMI dictionary (keyword -> score)
with open(PMI_PATH, "r", encoding="utf-8") as f:
    pmi_dict = json.load(f)

# Pragmatic labels
prag_rows = read_jsonl(PRAG_PATH)

def extract_labels(pragmatic_styles: Any) -> List[str]:
    labels = []
    if isinstance(pragmatic_styles, list):
        for item in pragmatic_styles:
            if isinstance(item, dict):
                labels.extend([str(k) for k in item.keys()])
            elif isinstance(item, str):
                labels.append(item)
    return labels

response_to_labels: Dict[str, List[str]] = {}
label_counter = Counter()
for row in prag_rows:
    resp = str(row.get("response", "")).strip()
    labels = extract_labels(row.get("pragmatic_styles", []))
    if not resp or not labels:
        continue
    response_to_labels.setdefault(resp, [])
    response_to_labels[resp].extend(labels)
    label_counter.update(labels)

# Select the top-50 most frequent labels as label space
STYLE_LABELS_50 = [k for k, _ in label_counter.most_common(50)]

# PCFG parse trees
with open(CONS_PATH, "r", encoding="utf-8") as f:
    cons_blocks = json.load(f)

flatten_trees = []
for block in cons_blocks:
    if isinstance(block, dict):
        flatten_trees.extend(block.get("con", []))

print(f"Muice texts: {len(muice_texts)}")
print(f"PMI keywords: {len(pmi_dict)}")
print(f"Pragmatic rows: {len(prag_rows)}, unique label count: {len(label_counter)}")
print(f"Constituency trees: {len(flatten_trees)}")
print(f"Top-50 labels example: {STYLE_LABELS_50[:10]}")

Muice texts: 2206
PMI keywords: 968
Pragmatic rows: 3502, unique label count: 45
Constituency trees: 2798
Top-50 labels example: ['cute', 'energetic', 'playful', 'kind', 'rational', 'curious', 'optimistic', 'hesitant', 'serious', 'tsundere']


## 3. Lexical Style Vector: Top-K Keyword Embedding Centroid

Select Top-K keywords by PMI × term frequency from sampled texts, then encode them with BGE and use the mean embedding as the lexical centroid.

In [12]:
TOP_K_LEXICAL = 30
PMI_CANDIDATE_POOL = 1000

model = SentenceTransformer(BGE_MODEL_NAME, device=device)
_dim = model.get_sentence_embedding_dimension()
emb_dim = int(_dim) if _dim is not None else 1024
print(f"BGE embedding dim: {emb_dim}")

# Cache high-PMI candidate keywords
pmi_sorted_keywords = [
    k for k, _ in sorted(pmi_dict.items(), key=lambda x: float(x[1]), reverse=True)
]
pmi_candidate_keywords = pmi_sorted_keywords[:PMI_CANDIDATE_POOL]


def l2_normalize(vec: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    norm = np.linalg.norm(vec)
    if norm < eps:
        return vec
    return vec / norm


def centroid_embeddings(text_units: List[str]) -> np.ndarray:
    if not text_units:
        return np.zeros(emb_dim, dtype=np.float32)
    embs = model.encode(
        text_units,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return l2_normalize(np.mean(embs, axis=0).astype(np.float32))


def select_topk_keywords_from_texts(
    texts: List[str],
    candidate_keywords: List[str],
    pmi_score_dict: Dict[str, float],
    top_k: int = TOP_K_LEXICAL,
) -> List[str]:
    scored = []
    for kw in candidate_keywords:
        tf = 0
        for text in texts:
            tf += text.count(kw)
        if tf > 0:
            score = tf * float(pmi_score_dict.get(kw, 0.0))
            scored.append((kw, score))

    if not scored:
        return candidate_keywords[:top_k]

    scored.sort(key=lambda x: x[1], reverse=True)
    return [kw for kw, _ in scored[:top_k]]


def lexical_vector_from_texts(texts: List[str]) -> Tuple[np.ndarray, List[str]]:
    keywords = select_topk_keywords_from_texts(
        texts, pmi_candidate_keywords, pmi_dict, top_k=TOP_K_LEXICAL
    )
    vec = centroid_embeddings(keywords)
    return vec, keywords

BGE embedding dim: 1024


## 4. Pragmatic Style Vector: Embedding Centroid of 50 Style Labels

Pre-encode the 50 style-label texts, then average embeddings of the activated label set to build the pragmatic centroid.

In [13]:
label_emb_table = model.encode(
    STYLE_LABELS_50,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)
label_to_idx = {label: i for i, label in enumerate(STYLE_LABELS_50)}


def activated_labels_from_texts(texts: List[str]) -> List[str]:
    labels = set()
    for text in texts:
        for lb in response_to_labels.get(text, []):
            if lb in label_to_idx:
                labels.add(lb)
    return sorted(labels)


def pragmatic_vector_from_labels(labels: List[str]) -> np.ndarray:
    if not labels:
        return np.zeros(emb_dim, dtype=np.float32)
    idx = [label_to_idx[lb] for lb in labels if lb in label_to_idx]
    if not idx:
        return np.zeros(emb_dim, dtype=np.float32)
    vec = np.mean(label_emb_table[idx], axis=0).astype(np.float32)
    return l2_normalize(vec)


def pragmatic_vector_from_texts(texts: List[str]) -> Tuple[np.ndarray, List[str]]:
    labels = activated_labels_from_texts(texts)
    vec = pragmatic_vector_from_labels(labels)
    return vec, labels

## 5. Syntactic Style Vector: 13-Dimensional PCFG Distribution

Count PCFG production-rule frequencies from full Muice parse trees, then use the top 13 rules to build a continuous probability vector.

In [14]:
PCFG_DIM = 13


def iter_rules_from_tree(tree: Any):
    if not (isinstance(tree, list) and len(tree) == 2):
        return
    lhs, rhs = tree
    if isinstance(rhs, list) and all(isinstance(ch, list) and len(ch) == 2 for ch in rhs):
        rhs_symbols = [str(ch[0]) for ch in rhs]
        rule = f"{lhs}->{' '.join(rhs_symbols)}"
        yield rule
        for ch in rhs:
            yield from iter_rules_from_tree(ch)


def collect_rule_counter(trees: List[Any]) -> Counter:
    cnt = Counter()
    for t in trees:
        for r in iter_rules_from_tree(t):
            cnt[r] += 1
    return cnt


full_rule_counter = collect_rule_counter(flatten_trees)
RULE_VOCAB_13 = [r for r, _ in full_rule_counter.most_common(PCFG_DIM)]


def syntactic_vector_from_trees(trees: List[Any], rule_vocab: List[str]) -> np.ndarray:
    cnt = collect_rule_counter(trees)
    vec = np.array([cnt.get(r, 0.0) for r in rule_vocab], dtype=np.float32)
    s = float(vec.sum())
    if s > 0:
        vec = vec / s
    return vec


def check_syntactic_vector(vec: np.ndarray):
    assert vec.shape[0] == PCFG_DIM, f"Syntactic vector dimension must be {PCFG_DIM}"
    assert np.all(vec >= -1e-9), "Syntactic vector must be non-negative"


print("Top-13 PCFG rules:")
for i, r in enumerate(RULE_VOCAB_13, 1):
    print(f"{i:02d}. {r}")

Top-13 PCFG rules:
01. ADVP->AD
02. NP->NN
03. NP->PN
04. IP->VP
05. VP->ADVP VP
06. VP->VV
07. IP->NP VP
08. VP->VV NP
09. VP->VV VP
10. CP->IP SP
11. TOP->IP
12. VP->VA
13. NP->DNP NP


Note: `outputs/cons/muice.json` provides constituency parse trees rather than pre-aggregated per-sample 13D features. Therefore, this experiment defines the syntactic vector as normalized frequencies over the top-13 PCFG rules from the full corpus, which can be compared directly with cosine similarity.

## 6. Unified Style Vector $S$ and Cosine Similarity

Define $S=[L;P;Y]$, implement cosine similarity, and build the full-corpus ground-truth vector $S_{full}$.

In [15]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    if np.linalg.norm(a) < 1e-12 or np.linalg.norm(b) < 1e-12:
        return 0.0
    return float(cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0])


def build_style_vector(texts: List[str], trees: List[Any]) -> Dict[str, Any]:
    lexical_vec, lexical_keywords = lexical_vector_from_texts(texts)
    pragmatic_vec, pragmatic_labels = pragmatic_vector_from_texts(texts)
    syntactic_vec = syntactic_vector_from_trees(trees, RULE_VOCAB_13)
    check_syntactic_vector(syntactic_vec)

    s_vec = np.concatenate([lexical_vec, pragmatic_vec, syntactic_vec]).astype(np.float32)
    return {
        "lexical_vec": lexical_vec,
        "pragmatic_vec": pragmatic_vec,
        "syntactic_vec": syntactic_vec,
        "S": s_vec,
        "lexical_keywords": lexical_keywords,
        "pragmatic_labels": pragmatic_labels,
    }


# Full-corpus reference vector
S_full_pack = build_style_vector(muice_texts, flatten_trees)
S_full = S_full_pack["S"]

print("S_full dims:")
print("  lexical:", S_full_pack["lexical_vec"].shape)
print("  pragmatic:", S_full_pack["pragmatic_vec"].shape)
print("  syntactic:", S_full_pack["syntactic_vec"].shape)
print("  total S:", S_full.shape)
print("full lexical keywords:", S_full_pack["lexical_keywords"][:10])
print("full pragmatic labels:", S_full_pack["pragmatic_labels"][:10])

S_full dims:
  lexical: (1024,)
  pragmatic: (1024,)
  syntactic: (13,)
  total S: (2061,)
full lexical keywords: ['喵', '好', '沐沐', '看', '没', '~', '天', '吃', '多', '才']
full pragmatic labels: ['airhead', 'anxious', 'cautious', 'chuunibyou', 'clingy', 'confident', 'conservative', 'controlling', 'curious', 'cute']


## 7. N-Shot Random Sampling Experiment ($N=\{10,25,50,100,200\}$)

For each $N$, draw one random sample and compute cosine similarity between the sampled style vector and $S_{full}$.

In [16]:
N_LIST = [10, 25, 50, 100, 200]


def sample_once(n: int, seed: int) -> Dict[str, float]:
    rng = np.random.default_rng(seed)

    text_idx = rng.choice(len(muice_texts), size=n, replace=False)
    sampled_texts = [muice_texts[i] for i in text_idx]

    # Tree count and text count come from different sources; sample trees independently
    tree_idx = rng.choice(len(flatten_trees), size=n, replace=False)
    sampled_trees = [flatten_trees[i] for i in tree_idx]

    s_pack = build_style_vector(sampled_texts, sampled_trees)
    return {
        "cos_lexical": cosine_sim(s_pack["lexical_vec"], S_full_pack["lexical_vec"]),
        "cos_pragmatic": cosine_sim(s_pack["pragmatic_vec"], S_full_pack["pragmatic_vec"]),
        "cos_syntactic": cosine_sim(s_pack["syntactic_vec"], S_full_pack["syntactic_vec"]),
        "cos_S": cosine_sim(s_pack["S"], S_full),
    }


single_run_rows = []
base_seed = 2026
for n in N_LIST:
    out = sample_once(n, seed=base_seed + n)
    single_run_rows.append({"N": n, **out})

single_run_df = pd.DataFrame(single_run_rows)
display(single_run_df)

,N,cos_lexical,cos_pragmatic,cos_syntactic,cos_S
0,10,0.970788,0.979704,0.932213,0.973044
1,25,0.983299,0.977255,0.989819,0.980727
2,50,0.988273,0.992966,0.995641,0.990852
3,100,0.988724,0.993941,0.995698,0.991533
4,200,0.993798,0.996235,0.997954,0.995152


## 8. Repeated Sampling Statistics (Mean / Variance / 95% CI)

Repeat sampling for each $N$, summarize mean, standard deviation, and 95% confidence interval, then export CSV files.

In [ ]:
REPEATS = 30

all_rows = []
for n in tqdm(N_LIST, desc="N-shot"):
    for r in range(REPEATS):
        seed = 10000 + n * 100 + r
        out = sample_once(n, seed=seed)
        all_rows.append({"N": n, "repeat": r, "seed": seed, **out})

all_df = pd.DataFrame(all_rows)


def summarize_metric(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    grp = df.groupby("N")[metric]
    summary = grp.agg(["mean", "std", "count"]).reset_index()
    summary["se"] = summary["std"] / np.sqrt(summary["count"])
    summary["ci95_low"] = summary["mean"] - 1.96 * summary["se"]
    summary["ci95_high"] = summary["mean"] + 1.96 * summary["se"]
    summary["metric"] = metric
    return summary

summary_df = pd.concat(
    [
        summarize_metric(all_df, "cos_lexical"),
        summarize_metric(all_df, "cos_pragmatic"),
        summarize_metric(all_df, "cos_syntactic"),
        summarize_metric(all_df, "cos_S"),
    ],
    ignore_index=True,
)

detail_csv = RESULT_ROOT / "nshot_similarity_detail.csv"
summary_csv = RESULT_ROOT / "nshot_similarity_summary.csv"
all_df.to_csv(detail_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("Saved:")
print("-", detail_csv)
print("-", summary_csv)

display(summary_df.head(10))

## 9. Convergence Plot and Automatic Convergence-Point Detection

Plot mean cosine similarity curves across $N$ (with confidence bands), and automatically detect convergence using threshold-based criteria.

In [ ]:
plot_metrics = [
    ("cos_S", "S (overall)", "#1f77b4"),
    ("cos_lexical", "Lexical", "#2ca02c"),
    ("cos_pragmatic", "Pragmatic", "#ff7f0e"),
    ("cos_syntactic", "Syntactic", "#d62728"),
]

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

for metric, label, color in plot_metrics:
    sdf = summary_df[summary_df["metric"] == metric].sort_values("N")
    x = sdf["N"].to_numpy()
    y = sdf["mean"].to_numpy()
    lo = sdf["ci95_low"].to_numpy()
    hi = sdf["ci95_high"].to_numpy()

    ax.plot(x, y, marker="o", label=label, color=color, linewidth=2)
    ax.fill_between(x, lo, hi, alpha=0.15, color=color)

ax.set_title("Muice N-Shot Stability: Cosine Similarity to $S_{full}$")
ax.set_xlabel("N-shot sample size")
ax.set_ylabel("Cosine similarity")

# Dynamic lower bound for better visibility while preserving 0 as a hard floor
y_lower = max(0.0, summary_df["ci95_low"].min() - 0.02)
ax.set_ylim(y_lower, 1.02)

ax.legend()

# Convergence criterion: mean >= tau and incremental gain < eps
tau = 0.95
eps = 0.005
s_df = summary_df[summary_df["metric"] == "cos_S"].sort_values("N").reset_index(drop=True)

n_vals = s_df["N"].to_numpy()
mean_vals = s_df["mean"].to_numpy()

conv_n = None
for i in range(1, len(n_vals)):
    gain = mean_vals[i] - mean_vals[i - 1]
    if mean_vals[i] >= tau and abs(gain) < eps:
        conv_n = int(n_vals[i])
        break

if conv_n is not None:
    conv_y = float(mean_vals[n_vals == conv_n][0])
    ax.axvline(conv_n, color="black", linestyle="--", linewidth=1)
    ax.scatter([conv_n], [conv_y], color="black", zorder=5)
    ax.text(conv_n + 2, conv_y - 0.04, f"Converged at N={conv_n}", fontsize=10)

plot_path_png = RESULT_ROOT / "nshot_convergence.png"
plot_path_pdf = RESULT_ROOT / "nshot_convergence.pdf"
fig.tight_layout()
fig.savefig(plot_path_png, bbox_inches="tight", dpi=300)
fig.savefig(plot_path_pdf, bbox_inches="tight")
plt.show()

print(f"Convergence N: {conv_n}")
print(f"Saved figure (PNG): {plot_path_png}")
print(f"Saved figure (PDF): {plot_path_pdf}")